# TabICLv2 Classifier — standalone Google Colab

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-classifier-pipeline/blob/main/tutorials/tabiclv2_classifier_colab.ipynb)

Run the exact pinned **TabICLv2 Classifier** without DIMER Workbench. The notebook supports a DIMER ZIP/direct checkpoint or the pinned upstream checkpoint, sample/BYOD data, pretrained in-context evaluation, optional GPU fine-tuning, inference, and export of a DIMER-style serving bundle.

TabICL remains an **in-context learner** after fine-tuning: the deployable model is **checkpoint + training context + manifest**, not the checkpoint alone.

> Load checkpoints and bundles only from sources you trust. Safe ZIP extraction does not make PyTorch checkpoint deserialization trustworthy.


## 1. Install and inspect the runtime


In [ ]:
%pip -q install "tabicl[finetune]==2.1.1" "pyarrow>=15" "pandas>=2" "scikit-learn>=1.4" "huggingface_hub>=0.25"

import importlib.metadata, sys, torch
TABICL_VERSION = "2.1.1"
if importlib.metadata.version("tabicl") != TABICL_VERSION:
    raise RuntimeError("Unexpected tabicl version")
print("Python:", sys.version.split()[0])
print("TabICL:", TABICL_VERSION)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 2. Acquire and SHA-256-verify the exact classifier checkpoint

Use **Pinned upstream** by default. **DIMER ZIP** accepts one `.ckpt` file or a ZIP containing exactly one `.ckpt` **only when it is the same pinned base checkpoint**; its bytes must match the pinned release. Fine-tuned DIMER serving bundles belong in the artifact-inference notebook, not this base-checkpoint acquisition step.


In [ ]:
from pathlib import Path
from google.colab import files
from huggingface_hub import hf_hub_download
import hashlib, shutil, stat, zipfile

MODEL_REPO = "jingang/TabICL"
CHECKPOINT_NAME = "tabicl-classifier-v2-20260212.ckpt"
MODEL_REVISION = "4dcd344ece2c00be9e831fdd35bed57b5ad83e19"
CHECKPOINT_SHA256 = "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"
CHECKPOINT_SOURCE = "Pinned upstream"  # @param ["Pinned upstream", "DIMER ZIP"]
WORK_DIR = Path("/content/tabiclv2-classifier")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_single_ckpt(zip_path, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        infos = []
        root = dest.resolve()
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            parts = Path(name).parts
            mode = info.external_attr >> 16
            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):
                raise ValueError(f"Unsafe ZIP member: {info.filename}")
            if info.is_dir():
                continue
            out = (dest / Path(name)).resolve()
            if root != out and root not in out.parents:
                raise ValueError(f"ZIP member escapes destination: {info.filename}")
            if Path(name).suffix.lower() == ".ckpt":
                infos.append(info)
        if len(infos) != 1:
            raise ValueError(f"Expected exactly one .ckpt, found {len(infos)}")
        info = infos[0]
        out = (dest / Path(info.filename)).resolve()
        out.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, out.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        return out

if CHECKPOINT_SOURCE == "Pinned upstream":
    checkpoint_path = Path(hf_hub_download(
        repo_id=MODEL_REPO, filename=CHECKPOINT_NAME,
        revision=MODEL_REVISION, local_dir=str(WORK_DIR / "pinned")))
elif CHECKPOINT_SOURCE == "DIMER ZIP":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one .ckpt or .zip")
    name, payload = next(iter(uploaded.items()))
    p = WORK_DIR / Path(name).name
    p.write_bytes(payload)
    checkpoint_path = p if p.suffix.lower() == ".ckpt" else safe_single_ckpt(p, WORK_DIR / "dimer")
else:
    raise ValueError(f"Unsupported CHECKPOINT_SOURCE: {CHECKPOINT_SOURCE}")

observed = sha256_file(checkpoint_path)
if observed != CHECKPOINT_SHA256:
    raise RuntimeError(f"Checkpoint SHA-256 mismatch: {observed}")
BASE_CHECKPOINT_PATH = checkpoint_path.resolve()
print("✓ Verified:", BASE_CHECKPOINT_PATH)
print("✓ SHA-256:", observed)


## 3. Load sample or BYOD data

The built-in sample is scikit-learn's Breast Cancer Wisconsin dataset. BYOD modes reject duplicate raw CSV headers, require ≥50 labelled training rows and ≥2 classes, align feature order, and reject evaluation labels unseen in training. Categorical/string/bool features are ordinal-encoded with a training-fitted map to match the DIMER fine-tuner's current workaround for TabICLv2 fine-tuning.

For temporal/grouped/leakage-sensitive data, create explicit partitions externally and use the pre-split mode.


In [ ]:
import csv, io, math, numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

DATA_SOURCE = "Sample: Breast Cancer"  # @param ["Sample: Breast Cancer", "Upload CSV", "Upload pre-split train/val/test"]
TARGET_COLUMN = "target"
VALIDATION_SPLIT = 0.20
RANDOM_SEED = 42
MIN_TRAIN_ROWS, MIN_EVAL_ROWS = 50, 2
MAX_TRAIN_ROWS, MAX_FEATURES = 50_000, 2_000

def raw_header(payload):
    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    for row in reader:
        if row and any(x.strip() for x in row):
            return row
    raise ValueError("CSV has no header")

def duplicate_names(names):
    seen, dupes = set(), []
    for name in names:
        if name in seen and name not in dupes: dupes.append(name)
        seen.add(name)
    return dupes

def read_csv_payload(payload, label):
    dupes = duplicate_names(raw_header(payload))
    if dupes: raise ValueError(f"{label} contains duplicate column names: {dupes}")
    return pd.read_csv(io.BytesIO(payload))

def prepare(frame, label, min_rows):
    if TARGET_COLUMN not in frame.columns: raise KeyError(f"{label}: missing target {TARGET_COLUMN!r}")
    before = len(frame)
    out = frame.dropna(subset=[TARGET_COLUMN]).copy().reset_index(drop=True)
    if before != len(out): print(f"⚠ {label}: dropped {before-len(out)} missing-target row(s)")
    if len(out) < min_rows: raise ValueError(f"{label}: need at least {min_rows} labelled rows")
    return out

def fit_encoder(frame, features):
    out = {}
    for col in features:
        s = frame[col]
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s): continue
        out[col] = sorted({str(v) for v in s.dropna().unique()})
    return out

def apply_encoder(frame, encoders):
    out = frame.copy()
    for col, cats in encoders.items():
        lookup, unknown = {c:i for i,c in enumerate(cats)}, len(cats)
        out[col] = [unknown if pd.isna(v) else lookup.get(str(v), unknown) for v in out[col]]
    return out

test_data = None
if DATA_SOURCE == "Sample: Breast Cancer":
    ds = load_breast_cancer(as_frame=True)
    frame = ds.frame.rename(columns={ds.target.name: TARGET_COLUMN})
    train_data, temp = train_test_split(frame, test_size=.4, random_state=RANDOM_SEED, stratify=frame[TARGET_COLUMN])
    holdout_data, test_data = train_test_split(temp, test_size=.5, random_state=RANDOM_SEED, stratify=temp[TARGET_COLUMN])
elif DATA_SOURCE == "Upload CSV":
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Upload exactly one CSV")
    name, payload = next(iter(uploaded.items()))
    frame = prepare(read_csv_payload(payload, name), name, MIN_TRAIN_ROWS)
    counts = frame[TARGET_COLUMN].value_counts()
    if counts.size < 2 or counts.min() < 2: raise ValueError("Every class needs ≥2 rows for stratified holdout")
    train_data, holdout_data = train_test_split(frame, test_size=VALIDATION_SPLIT, random_state=RANDOM_SEED, stratify=frame[TARGET_COLUMN])
else:
    uploaded = files.upload()
    by_name = {Path(k).name.lower():(k,v) for k,v in uploaded.items()}
    if not {"train.csv","val.csv"} <= set(by_name): raise ValueError("Upload train.csv and val.csv; test.csv optional")
    n,p = by_name["train.csv"]; train_data = prepare(read_csv_payload(p,n), n, MIN_TRAIN_ROWS)
    n,p = by_name["val.csv"]; holdout_data = prepare(read_csv_payload(p,n), n, MIN_EVAL_ROWS)
    if "test.csv" in by_name:
        n,p = by_name["test.csv"]; test_data = prepare(read_csv_payload(p,n), n, MIN_EVAL_ROWS)

train_data = prepare(train_data, "train", MIN_TRAIN_ROWS)
holdout_data = prepare(holdout_data, "holdout", MIN_EVAL_ROWS)
if test_data is not None: test_data = prepare(test_data, "test", MIN_EVAL_ROWS)
FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
if not FEATURE_COLUMNS: raise ValueError("No feature columns")
if len(FEATURE_COLUMNS) > MAX_FEATURES or len(train_data) > MAX_TRAIN_ROWS: raise ValueError("Operational row/feature ceiling exceeded")
TRAIN_CLASSES = set(train_data[TARGET_COLUMN])
if len(TRAIN_CLASSES) < 2: raise ValueError("Need at least 2 training classes")

def align(frame, label):
    expected = set(FEATURE_COLUMNS + [TARGET_COLUMN])
    if set(frame.columns) != expected: raise ValueError(f"{label} schema does not match train")
    unseen = sorted(set(frame[TARGET_COLUMN]) - TRAIN_CLASSES)
    if unseen: raise ValueError(f"{label} has target classes unseen in training: {unseen}")
    return frame[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)

train_data = train_data[FEATURE_COLUMNS + [TARGET_COLUMN]].reset_index(drop=True)
holdout_data = align(holdout_data, "holdout")
test_data = align(test_data, "test") if test_data is not None else None
CATEGORICAL_ENCODERS = fit_encoder(train_data, FEATURE_COLUMNS)
train_encoded = apply_encoder(train_data, CATEGORICAL_ENCODERS)
holdout_encoded = apply_encoder(holdout_data, CATEGORICAL_ENCODERS)
test_encoded = apply_encoder(test_data, CATEGORICAL_ENCODERS) if test_data is not None else None
print(f"✓ train={len(train_encoded)}, holdout={len(holdout_encoded)}, test={0 if test_encoded is None else len(test_encoded)}")
print(f"✓ features={len(FEATURE_COLUMNS)}, classes={len(TRAIN_CLASSES)}")


## 4. Evaluate pretrained TabICLv2, then optionally fine-tune

The committed default is **no fine-tuning**. Fine-tuning requires CUDA, uses the upstream `FinetunedTabICLClassifier`, and writes `best.ckpt`. For a fair comparison, that checkpoint is reloaded into the ordinary classifier with the same inference ensemble as the pretrained baseline. The independent test split is never used for model selection. In **Upload CSV** mode, no independent test split is created: the generated holdout is used for both reporting and pretrained-vs-fine-tuned selection, so treat those metrics as selection-biased rather than independent test evidence.


In [ ]:
from tabicl import TabICLClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss, roc_auc_score

RUN_FINE_TUNING = False  # @param {type:"boolean"}
EVAL_METRIC = "accuracy"  # @param ["accuracy", "log_loss", "roc_auc"]
N_ESTIMATORS = 8
FINE_TUNE_EPOCHS, FINE_TUNE_TIME_LIMIT, FINE_TUNE_PATIENCE = 10, 600, 3
MIN_SELECTION_HOLDOUT_ROWS = 50
if EVAL_METRIC not in {"accuracy","log_loss","roc_auc"}: raise ValueError(f"Unsupported EVAL_METRIC: {EVAL_METRIC}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
X_train, y_train = train_encoded[FEATURE_COLUMNS], train_encoded[TARGET_COLUMN]

def metrics(model, frame):
    X, y = frame[FEATURE_COLUMNS], frame[TARGET_COLUMN]
    pred, proba = np.asarray(model.predict(X)), np.asarray(model.predict_proba(X))
    out = {
        "accuracy": float(accuracy_score(y,pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y,pred)),
        "f1_weighted": float(f1_score(y,pred,average="weighted")),
        "log_loss": float(log_loss(y,proba,labels=model.classes_)),
    }
    try:
        out["roc_auc"] = float(roc_auc_score(y, proba[:,1])) if len(model.classes_) == 2 else float(roc_auc_score(y,proba,multi_class="ovr",labels=model.classes_))
    except ValueError:
        out["roc_auc"] = float("nan")
    return out

def better(a,b,name):
    av,bv = a[name],b[name]
    if not (math.isfinite(av) and math.isfinite(bv)): raise ValueError(f"{name} unavailable for selection")
    return av < bv if name == "log_loss" else av > bv

def show(label, m):
    print(label, {k:(round(v,6) if math.isfinite(v) else None) for k,v in m.items()})

baseline_model = TabICLClassifier(model_path=str(BASE_CHECKPOINT_PATH), allow_auto_download=False, n_estimators=N_ESTIMATORS, random_state=RANDOM_SEED, device=DEVICE, support_many_classes=True)
baseline_model.fit(X_train,y_train)
baseline_metrics = metrics(baseline_model, holdout_encoded)
baseline_test_metrics = metrics(baseline_model, test_encoded) if test_encoded is not None else None
if not math.isfinite(baseline_metrics[EVAL_METRIC]):
    raise ValueError(
        f"{EVAL_METRIC} is unavailable on the holdout before fine-tuning; "
        "choose another selection metric or provide a holdout with sufficient class coverage"
    )
show("Pretrained holdout", baseline_metrics)
if baseline_test_metrics: show("Pretrained test", baseline_test_metrics)

candidate_model = candidate_metrics = candidate_test_metrics = candidate_checkpoint = None
if RUN_FINE_TUNING:
    if not torch.cuda.is_available(): raise RuntimeError("TabICLv2 fine-tuning requires CUDA")
    from tabicl import FinetunedTabICLClassifier
    ft_dir = Path("/content/tabiclv2-classifier-finetune")
    if ft_dir.exists(): shutil.rmtree(ft_dir)
    finetuner = FinetunedTabICLClassifier(
        epochs=FINE_TUNE_EPOCHS, learning_rate=1e-5, weight_decay=.01,
        n_estimators_finetune=1, n_estimators_validation=1, n_estimators_inference=4,
        early_stopping=True, patience=FINE_TUNE_PATIENCE, time_limit=FINE_TUNE_TIME_LIMIT,
        eval_metric=EVAL_METRIC, model_path=str(BASE_CHECKPOINT_PATH), allow_auto_download=False,
        device="cuda", random_state=RANDOM_SEED, verbose=True, support_many_classes=True)
    finetuner.fit(X_train,y_train,X_val=holdout_encoded[FEATURE_COLUMNS],y_val=holdout_encoded[TARGET_COLUMN],output_dir=str(ft_dir))
    candidate_checkpoint = ft_dir/"best.ckpt"
    if not candidate_checkpoint.exists(): raise RuntimeError("Fine-tuning did not produce best.ckpt")
    candidate_model = TabICLClassifier(model_path=str(candidate_checkpoint),allow_auto_download=False,n_estimators=N_ESTIMATORS,random_state=RANDOM_SEED,device=DEVICE,support_many_classes=True)
    candidate_model.fit(X_train,y_train)
    candidate_metrics = metrics(candidate_model, holdout_encoded)
    candidate_test_metrics = metrics(candidate_model,test_encoded) if test_encoded is not None else None
    show("Fine-tuned holdout",candidate_metrics)
    if candidate_test_metrics: show("Fine-tuned test",candidate_test_metrics)

ACTIVE_MODEL, ACTIVE_CHECKPOINT_PATH, ACTIVE_MODE = baseline_model, BASE_CHECKPOINT_PATH, "pretrained"
SELECTION_BASIS = "default:pretrained"
if candidate_model is not None:
    if len(holdout_encoded) < MIN_SELECTION_HOLDOUT_ROWS:
        SELECTION_BASIS = f"default:pretrained; holdout-too-small:{len(holdout_encoded)}<{MIN_SELECTION_HOLDOUT_ROWS}"
        print("⚠ Holdout too small for automatic selection; keeping pretrained")
    else:
        SELECTION_BASIS = f"holdout:{EVAL_METRIC}"
        if better(candidate_metrics,baseline_metrics,EVAL_METRIC):
            ACTIVE_MODEL, ACTIVE_CHECKPOINT_PATH, ACTIVE_MODE = candidate_model, candidate_checkpoint, "fine-tuned"
    print(f"✓ Recommended for export: {ACTIVE_MODE} ({SELECTION_BASIS})")
else:
    print("✓ Recommended for export: pretrained (fine-tuning not run)")


## 5. Optional new-data inference

Upload one CSV. Extra columns are preserved; existing `prediction`/`probability_*` columns are rejected.


In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}

def read_inference_csv(payload, feature_columns):
    frame = read_csv_payload(payload, "Inference CSV")
    if "prediction" in frame.columns or any(c.startswith("probability_") for c in frame.columns):
        raise ValueError("Inference CSV already contains prediction/probability output columns")
    missing = [c for c in feature_columns if c not in frame.columns]
    if missing: raise ValueError(f"Inference CSV missing features: {missing}")
    return frame

if RUN_NEW_DATA_INFERENCE:
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Upload exactly one CSV")
    _, payload = next(iter(uploaded.items()))
    rows = read_inference_csv(payload, FEATURE_COLUMNS)
    X = apply_encoder(rows[FEATURE_COLUMNS], CATEGORICAL_ENCODERS)
    pred, proba = np.asarray(ACTIVE_MODEL.predict(X)), np.asarray(ACTIVE_MODEL.predict_proba(X))
    out = rows.copy(); out["prediction"] = pred
    for i,cls in enumerate(ACTIVE_MODEL.classes_): out[f"probability_{cls}"] = proba[:,i]
    out_path = Path("/content/tabiclv2_classifier_predictions.csv")
    out.to_csv(out_path,index=False); files.download(str(out_path))
else:
    print("Inference skipped.")


## 6. Export a DIMER-style portable bundle

The ZIP contains the same minimum serving contract as the DIMER pipeline: `checkpoints/best.ckpt`, `training_context.parquet`, and `artifact.json`. The training context is required because TabICL is still an in-context learner at serve time.

> **Data governance:** this serving bundle embeds the labelled training context. Treat the ZIP with the same licence, access-control, retention, and disclosure rules as the source training dataset.


In [ ]:
import json
ARTIFACT_DIR = Path("/content/tabicl_classifier")
if ARTIFACT_DIR.exists(): shutil.rmtree(ARTIFACT_DIR)
(ARTIFACT_DIR/"checkpoints").mkdir(parents=True)
export_ckpt = ARTIFACT_DIR/"checkpoints"/"best.ckpt"
shutil.copy2(ACTIVE_CHECKPOINT_PATH, export_ckpt)
context_path = ARTIFACT_DIR/"training_context.parquet"
train_encoded[FEATURE_COLUMNS+[TARGET_COLUMN]].to_parquet(context_path,index=False)
manifest = {
    "artifactFormat":"tabicl-dimer-classifier-v1",
    "checkpoint":"checkpoints/best.ckpt","trainingContext":"training_context.parquet",
    "targetColumn":TARGET_COLUMN,"featureColumns":FEATURE_COLUMNS,
    "baseCheckpoint":CHECKPOINT_NAME,"baseModelRevision":MODEL_REVISION,"baseModelSha256":CHECKPOINT_SHA256,
    "tabiclVersion":TABICL_VERSION,"mode":ACTIVE_MODE,"selectionBasis":SELECTION_BASIS,
    "inference":{"class":"TabICLClassifier","modelPath":"checkpoints/best.ckpt","nEstimators":N_ESTIMATORS,
                 "randomState":RANDOM_SEED,"supportManyClasses":True,"allowAutoDownload":False,
                 "categoricalEncoders":CATEGORICAL_ENCODERS},
    "digests":{"checkpointSha256":sha256_file(export_ckpt),"trainingContextSha256":sha256_file(context_path)},
    "aiProvenance":{"generatedWith":"GPT-5.6 Sol High","provider":"OpenAI / ChatGPT","agentRelayRole":"Builder","note":"Provenance only; not independent sign-off."}
}
(ARTIFACT_DIR/"artifact.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
archive_path = Path(shutil.make_archive("/content/tabiclv2-classifier-artifact","zip",root_dir=ARTIFACT_DIR))
print("✓ Artifact:",archive_path)
print("✓ ZIP SHA-256:",sha256_file(archive_path))
files.download(str(archive_path))


## 7. Fresh reload smoke test


In [ ]:
RELOAD_DIR = Path("/content/tabiclv2-classifier-reload")
if RELOAD_DIR.exists(): shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir()
with zipfile.ZipFile(archive_path) as z:
    root = RELOAD_DIR.resolve()
    for info in z.infolist():
        name=info.filename.replace("\\","/"); parts=Path(name).parts; mode=info.external_attr>>16
        if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode): raise ValueError(f"Unsafe artifact member: {info.filename}")
        out=(RELOAD_DIR/Path(name)).resolve()
        if root != out and root not in out.parents: raise ValueError("Artifact path escapes destination")
    z.extractall(RELOAD_DIR)
served=json.loads((RELOAD_DIR/"artifact.json").read_text())
ckpt=RELOAD_DIR/served["checkpoint"]; ctxp=RELOAD_DIR/served["trainingContext"]
if sha256_file(ckpt)!=served["digests"]["checkpointSha256"] or sha256_file(ctxp)!=served["digests"]["trainingContextSha256"]:
    raise RuntimeError("Artifact digest mismatch")
ctx=pd.read_parquet(ctxp)
reloaded=TabICLClassifier(model_path=str(ckpt),allow_auto_download=False,n_estimators=served["inference"]["nEstimators"],random_state=served["inference"]["randomState"],device=DEVICE,support_many_classes=True)
reloaded.fit(ctx[served["featureColumns"]],ctx[served["targetColumn"]])
smoke=holdout_encoded[FEATURE_COLUMNS].iloc[:min(8,len(holdout_encoded))]
if not np.array_equal(np.asarray(ACTIVE_MODEL.predict(smoke)),np.asarray(reloaded.predict(smoke))): raise RuntimeError("Prediction mismatch")
if not np.allclose(np.asarray(ACTIVE_MODEL.predict_proba(smoke)),np.asarray(reloaded.predict_proba(smoke)),rtol=1e-5,atol=1e-7): raise RuntimeError("Probability mismatch")
print("✓ Exported bundle reloads and reproduces smoke predictions.")


## AI provenance

This tutorial was developed with substantial AI assistance using **GPT-5.6 Sol High**, via **OpenAI / ChatGPT**, under Agent Relay role **Builder**, with maintainer direction and review. Attribution is provenance, not sign-off or independent verification.

Upstream model/code: TabICLv2 / `tabicl`, Soda team at Inria, BSD-3-Clause.
